In [2]:
import logging
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import month, days
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, TimestampNTZType

In [3]:
def create_spark_session(app_name: str) -> SparkSession:
    spark = (
        SparkSession.builder
        .master("spark://spark-master:7077")
        .appName(app_name)
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.lakehouse_prod","org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.lakehouse_prod.type", "hive")
        .config("spark.sql.catalog.lakehouse_prod.uri","thrift://hive-metastore:9083")
        .config("spark.sql.catalog.lakehouse_prod.warehouse","s3a://lakehouse-prod-bucket/warehouse/")
        .config("spark.sql.catalog.lakehouse_prod.io-impl","org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .enableHiveSupport()
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")
    print("NOTE: SparkSession created successfully!")

    return spark

# Create Spakr Session
app_name = 'Spark-Iceberg-DB-Management'
spark = create_spark_session(app_name)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 17:04:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


NOTE: SparkSession created successfully!


In [4]:
spark.sql("""SHOW DATABASES;""").show()

+---------+
|namespace|
+---------+
|bronze_db|
|  default|
|  gold_db|
|silver_db|
+---------+



In [5]:
catalog = "lakehouse_prod"
schema = "bronze_db"
table = "bronze_currency_profiles"
script = f"""SELECT * FROM {catalog}.{schema}.{table};"""

spark.sql(script).show()

[Stage 0:>                                                          (0 + 1) / 1]

+-------------+------+--------------------+-------------+--------------+--------+----+--------------------+--------------------+
|currency_code|symbol|                name|symbol_native|decimal_digits|rounding|code|         name_plural|       Ingested_Time|
+-------------+------+--------------------+-------------+--------------+--------+----+--------------------+--------------------+
|          USD|     $|           US Dollar|            $|             2|     0.0| USD|          US dollars|2026-03-28 07:59:...|
|          CAD|   CA$|     Canadian Dollar|            $|             2|     0.0| CAD|    Canadian dollars|2026-03-28 07:59:...|
|          EUR|     €|                Euro|            €|             2|     0.0| EUR|               euros|2026-03-28 07:59:...|
|          AED|   AED|United Arab Emira...|        د.إ.‏|             2|     0.0| AED|         UAE dirhams|2026-03-28 07:59:...|
|          AFN|    Af|      Afghan Afghani|            ؋|             0|     0.0| AFN|     Afghan

In [5]:
# DROP TABLES
catalog = "lakehouse_prod"
schema = "bronze_db"
table = "bronze_currency_profiles"

spark.sql(f'''
DROP TABLE {catalog}.{schema}.{table};
''')


DataFrame[]

In [6]:
spark.stop()